# Milestone 4 : fiabiliser la solution et analyser les gains

## 1. Suites de tests

Les tests portent sur le moteur du milestone 3, découpé en modules dans le package `backend/` (`couts`, `jours_fixes`, `charge`, `contraintes`, ...), dont `main.py` enchaîne les étapes. Le notebook appelle `main.executer()` : le moteur tourne une fois (planning final, sorties 1 et 2), sans rien afficher ni écrire. Les tests appellent directement les fonctions des modules (`couts.cout_transport`, `jours_fixes.consolider_couple`, ...) et les résultats du moteur (`r.planning_complet`, `r.sortie2`, ...).

Chaque test est une petite fonction avec des `assert`. Le décorateur `@test` l'exécute tout de suite et note son résultat : OK, ÉCHEC (une vérification est fausse, avec le message) ou ERREUR (le code a planté). Le bilan de tous les tests est dans la partie 1.5.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import main
from backend import charge, config, contraintes, couts, donnees, formatage, jours_fixes

r = main.executer()   # le moteur tourne une fois (planning final, sorties 1 et 2), sans rien écrire ni afficher

RESULTATS = {}   # (section, test) -> résultat ; relancer une cellule remplace ses résultats


def test(section, description):
    # Décorateur : exécute le test tout de suite et note son résultat
    def executer(fonction):
        try:
            fonction()
            resultat, detail = "OK", ""
        except AssertionError as e:
            resultat, detail = "ÉCHEC", str(e)
        except Exception as e:
            resultat, detail = "ERREUR", f"{type(e).__name__} : {e}"
        RESULTATS[(section, description)] = {"Section": section, "Test": description, "Résultat": resultat, "Détail": detail}
        return fonction
    return executer


def resultats(section):
    # Résultats d'une section, dans l'ordre des tests (le détail d'un échec est affiché dans la partie 1.5)
    table = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == section]).drop(columns=["Section", "Détail"])
    return table.style.hide(axis="index").map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "",
                                              subset=["Résultat"])


J = pd.Timestamp
P, L = r.planning_complet, r.lignes_complet   # planning final (une ligne par camion) et lignes de besoin livrées
print(f"Moteur chargé : {len(P)} camions ({int(P['reserve'].sum())} réservés), {len(L)} lignes de besoin, "
      f"du {donnees.DEBUT:%d/%m} au {donnees.FIN:%d/%m}")

### 1.1 Tests unitaires des fonctions

Les fonctions clés du moteur sont testées sur des cas construits à la main, dont on connaît le résultat : la grille tarifaire, la date de départ des camions, la consolidation d'un couple et le contrôle du lissage. Les dates sont celles de la semaine du lundi 16/11/2026.

In [ ]:
ROLES = {
    "tranche": "Trouve la tranche de la grille tarifaire (distance ou poids) à appliquer : la première borne supérieure ou égale à la valeur",
    "cout_transport": "Calcule le prix d'un camion : coût de la grille × (1 + marge de 239,23 %) × surcharge diesel de 1,1943",
    "date_camion": "Choisit le jour de départ du camion qui livre un besoin : le dernier jour fixe du couple avant ce besoin",
    "profil_utilise": "Vérifie qu'un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours (C11)",
    "consolider_couple": "Regroupe les besoins d'un couple fournisseur × usine dans des camions, sans dépasser 24,5 t ni 110,4 m³ ni 7 jours d'avance",
    "hors_tolerance": "Contrôle du lissage : repère les semaines où une usine reçoit beaucoup plus de camions certains jours que d'autres",
}


def lignes_test(*lignes):
    # Lignes de besoin d'un couple fictif : (date du besoin, poids en kg, volume en m³)
    return pd.DataFrame([{"need_date": J(d), "weight": w, "volume": v} for d, w, v in lignes])


@test("1.1", "tranche : 30 km -> tranche 30 km ; 30,1 km -> tranche 50 km ; 651 km -> tranche > 650 km ; 4 000,5 kg -> tranche > 4 000 kg")
def _():
    assert couts.tranche(30, couts.TRANCHES_KM) == "30" and couts.tranche(30.1, couts.TRANCHES_KM) == "50"
    assert couts.tranche(651, couts.TRANCHES_KM) == ">650" and couts.tranche(4000.5, couts.TRANCHES_KG) == ">4000"


@test("1.1", "cout_transport : 300 km et 1 000 kg -> 335,34 €, le prix de l'exemple de la rate sheet")
def _():
    assert couts.cout_transport(1000, 300) == 335.34, couts.cout_transport(1000, 300)


@test("1.1", "date_camion : besoin le mercredi 18/11, livraison le lundi -> camion le lundi 16/11 ; besoin le vendredi 20/11, "
             "livraison lundi et jeudi -> camion le jeudi 19/11 ; besoin le lundi 09/11, livraison le jeudi -> camion le jeudi 05/11, "
             "dans la semaine ajoutée avant le planning")
def _():
    assert jours_fixes.date_camion(J("2026-11-18"), (0,)) == J("2026-11-16")
    assert jours_fixes.date_camion(J("2026-11-20"), (0, 3)) == J("2026-11-19")
    assert jours_fixes.date_camion(J("2026-11-09"), (3,)) == J("2026-11-05")


@test("1.1", "profil_utilise : profil lundi + jeudi, camions le lundi 16/11 et le jeudi 19/11 -> utilisé ; camions le lundi 16/11 "
             "et le jeudi 26/11 (jamais les deux la même semaine) -> pas utilisé ; profil d'un seul jour -> toujours utilisé")
def _():
    assert jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-19")], (0, 3))
    assert not jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-26")], (0, 3))
    assert jours_fixes.profil_utilise([J("2026-11-16"), J("2026-11-23")], (0,))


@test("1.1", "consolider_couple : 3 besoins de la semaine (1 t au total) -> 1 seul camion, le lundi ; 3 besoins de 10 t -> "
             "2 camions (30 t ne tiennent pas dans 24,5 t) ; 2 besoins de 60 m³ -> 2 camions (120 m³ > 110,4 m³)")
def _():
    semaine = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 500, 2), ("2026-11-17", 300, 1), ("2026-11-20", 200, 1)), (0,))
    assert len(semaine) == 1 and semaine["date"].iloc[0] == J("2026-11-16")
    lourd = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 10000, 1), ("2026-11-17", 10000, 1), ("2026-11-18", 10000, 1)), (0,))
    assert len(lourd) == 2 and (lourd["poids"] <= config.POIDS_MAX).all()
    volumineux = jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 60), ("2026-11-17", 100, 60)), (0,))
    assert len(volumineux) == 2 and (volumineux["volume"] <= config.VOLUME_MAX).all()


@test("1.1", "consolider_couple : un besoin 7 jours après le camion du lundi part dans ce camion ; à 8 jours, "
             "il part dans le camion de la semaine suivante")
def _():
    assert len(jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 1), ("2026-11-23", 100, 1)), (0,))) == 1
    assert len(jours_fixes.consolider_couple(lignes_test(("2026-11-16", 100, 1), ("2026-11-24", 100, 1)), (0,))) == 2


@test("1.1", "hors_tolerance : 2-2-2-2-2 camions du lundi au vendredi -> semaine équilibrée ; 10-0-0-0-0 -> hors tolérance ; "
             "3-2-2-2-1 -> équilibrée (écart d'un seul camion) ; 1 seul camion dans la semaine -> pas contrôlée")
def _():
    semaines = np.array([[2, 2, 2, 2, 2], [10, 0, 0, 0, 0], [3, 2, 2, 2, 1], [1, 0, 0, 0, 0]], dtype=float)
    assert charge.hors_tolerance(semaines).tolist() == [False, True, False, False]


# Tableau : la fonction, son rôle dans le moteur, le cas testé et le résultat
tests_11 = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == "1.1"])
tests_11[["Fonction", "Ce qu'on vérifie"]] = tests_11["Test"].str.split(" : ", n=1, expand=True)
tests_11["Rôle de la fonction"] = tests_11["Fonction"].map(ROLES)
(tests_11[["Fonction", "Rôle de la fonction", "Ce qu'on vérifie", "Résultat"]].style.hide(axis="index")
 .set_properties(**{"text-align": "left", "white-space": "normal"})
 .map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "", subset=["Résultat"]))

### 1.2 Tests du respect des contraintes

Les contraintes ont été définies au milestone 3 (partie 1 « Contraintes et objectif »). Chacune y a sa fonction de vérification, qui retourne les cas qui la violent (une table vide = contrainte respectée) :

| # | Contrainte | Fonction |
|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | `c4_un_couple` |
| C5 | Jamais de retard | `c5_retard` |
| C6 | Au plus 7 jours d'avance | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning (semaine ajoutée avant le planning comprise) | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 à 3 jours par couple, chaque camion part un de ces jours, sans exception | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, sauf besoin d'un jour > 24,5 t | `c9_un_camion_par_jour` |
| C10 | Lissage : aucun jour ne s'écarte de plus de 35 % de la moyenne de la semaine | `c10_lissage` |
| C11 | Profil utilisé : un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours fixes | `c11_profil_utilise` |

Au milestone 3 (partie 5), ces fonctions ont été appliquées au planning final : toutes les contraintes dures sont respectées et 2 semaines sur 20 restent hors tolérance pour le lissage.

Ici, on teste deux choses :
1. le planning final respecte toujours les contraintes : on rejoue les fonctions du milestone 3 sur le planning produit par `main.py` 

2. les fonctions de vérification détectent vraiment une erreur : une fonction qui renverrait toujours une table vide passerait le premier test sans rien vérifier. Pour chaque contrainte, on introduit donc volontairement une erreur dans une copie du planning (un camion de 24,5 t + 1 kg, une ligne livrée un jour en retard, un camion un samedi, ...) et on vérifie que la fonction la signale.

In [ ]:
FONCTIONS = {code_c: fonction for code_c, _, _, fonction in contraintes.CONTRAINTES}


def violations(code_c, planning, lignes):
    # Violations d'une contrainte (pour C9 : seulement les cas non justifiés par un besoin d'un jour > 24,5 t)
    args = (planning, lignes, donnees.IDS_ATTENDUS) if code_c == "C3" else (planning, lignes)
    v = FONCTIONS[code_c](*args)
    return v[~v["justifie"]] if code_c == "C9" else v


# ---------- Erreurs introduites volontairement, une par contrainte ----------
opti = P[~P["reserve"]]
i0 = opti.index[0]                                                          # un camion optimisé quelconque
multi = L.groupby("camion_id").size().loc[lambda n: n > 1].index[0]         # un camion qui transporte plusieurs lignes
couples_c9 = set(map(tuple, contraintes.c9_un_camion_par_jour(P, L)[["plant", "supplier_id"]].values))
i9 = next(i for i, r in opti.iterrows() if (r["plant"], r["supplier_id"]) not in couples_c9)
i8 = i0
i11 = next(i for i, prof in opti["profil"].items() if len(prof) == 1)        # un camion d'un couple à 1 jour fixe


def avec_erreur(code_c):
    # Copie du planning et des lignes avec une erreur qui viole la contrainte
    p, l = P.copy(), L.copy()
    if code_c == "C1":
        p.loc[i0, "poids"] = config.POIDS_MAX + 1                                    # camion de 24,5 t + 1 kg
    elif code_c == "C2":
        p.loc[i0, "volume"] = config.VOLUME_MAX + 1                                  # camion de 110,4 m³ + 1 m³
    elif code_c == "C3":
        l = l.drop(index=l.index[0])                                            # ligne oubliée
    elif code_c == "C4":
        l.loc[l.index[l["camion_id"] == multi][0], "supplier_id"] = "SUP_99999"  # ligne d'un autre fournisseur
    elif code_c == "C5":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] + pd.Timedelta(days=1)   # 1 jour de retard
    elif code_c == "C6":
        l.loc[l.index[0], "date_livraison"] = l.loc[l.index[0], "need_date"] - pd.Timedelta(days=8)   # 8 jours d'avance
    elif code_c == "C7":
        p.loc[i0, "date"] = J("2026-11-21")                                     # camion un samedi
    elif code_c == "C8":
        date, profil = p.loc[i8, "date"], p.loc[i8, "profil"]                   # camion hors de ses jours fixes
        p.loc[i8, "date"] = date + pd.Timedelta(days=next(j for j in range(5) if j not in profil) - date.day_of_week)
    elif code_c == "C9":
        p = pd.concat([p, p.loc[[i9]]], ignore_index=True)                      # 2 camions le même jour sans raison
    elif code_c == "C11":
        plant, supplier, jour = p.loc[i11, "plant"], p.loc[i11, "supplier_id"], p.loc[i11, "date"].day_of_week
        du_couple = (p["plant"] == plant) & (p["supplier_id"] == supplier)       # 2e jour fixe jamais utilisé
        p["profil"] = [tuple(sorted({jour, (jour + 1) % 5})) if m else prof for m, prof in zip(du_couple, p["profil"])]
    elif code_c == "C10":
        renfort = p[p["plant"] == donnees.USINES[0]].head(1).assign(date=J("2026-11-16"))
        p = pd.concat([p] + [renfort] * 15, ignore_index=True)                  # 15 camions de plus un lundi
    return p, l


@test("1.2", "Le planning final respecte les contraintes dures C1 à C9 et C11 ; C10 : au plus 2 semaines hors tolérance")
def _():
    nb = {c: len(violations(c, P, L)) for c, _, _, _ in contraintes.CONTRAINTES}
    violees = {c: n for c, n in nb.items() if c != "C10" and n > 0}
    assert not violees, f"contraintes violées : {violees}"
    assert nb["C10"] <= 2, f"{nb['C10']} semaines hors tolérance"


@test("1.2", "Chaque fonction de vérification (C1 à C11) détecte l'erreur qu'on lui introduit")
def _():
    ratees = [c for c, _, _, _ in contraintes.CONTRAINTES if len(violations(c, *avec_erreur(c))) <= len(violations(c, P, L))]
    assert not ratees, f"erreur non détectée par : {ratees}"


resultats("1.2")

### 1.3 Cas limites et hypothèses de données manquantes

Les hypothèses prises aux milestones 1 à 3 pour les données incomplètes et les cas particuliers sont vérifiées sur le planning final :
- ligne de plus de 24,5 t : exclue du moteur, en attendant la réponse du client ;
- poids manquant : la ligne ne peut pas être consolidée, elle a son propre camion réservé, le jour du besoin, chiffré au tarif > 4 000 kg 

- distance manquante : le camion est chiffré au tarif > 650 km, et la distance est affichée « inconnue » dans la sortie 1 

- début du planning : on peut livrer dans la semaine qui précède le planning (S45, à partir du lundi 02/11). Les premiers besoins partent donc, comme les autres, le dernier jour fixe qui les précède, au plus 7 jours avant.

- cas impossibles : une ligne sans poids garde toujours son camion réservé, le jour du besoin (règle absolue), même si le couple reçoit alors 2 camions ce jour-là ou si ce n'est pas un de ses jours fixes ; de même, un couple dont le besoin d'un jour dépasse 24,5 t reçoit 2 camions. Ces cas ne sont pas cachés : chacun est notifié dans les alertes (`data/gold/alertes.csv`), avec sa cause et l'action à mener.

In [ ]:
q = donnees.quarantaine


@test("1.3", "Ligne de plus de 24,5 t : exclue du moteur et absente des sorties")
def _():
    lourdes = q[q["weight"] > config.POIDS_MAX]
    assert len(lourdes) == 1, f"{len(lourdes)} ligne(s) de plus de 24,5 t"
    cles = ["need_date", "supplier_id", "material", "plant", "unloading_point"]
    assert lourdes[cles].merge(L[cles]).empty, "la ligne lourde est livrée"


@test("1.3", "Poids manquant : une ligne = un camion réservé, le jour du besoin, tarif > 4 000 kg")
def _():
    reserves, lignes_r = P[P["reserve"]], L[L["reserve"]]
    assert len(reserves) == q["weight"].isna().sum() == len(lignes_r)
    assert (lignes_r["date_livraison"] == lignes_r["need_date"]).all()
    assert np.allclose(reserves["cout"], [couts.cout_transport(couts.POIDS_PAR_DEFAUT, couts.distances[s]) for s in reserves["supplier_id"]])


@test("1.3", "Distance manquante : camions chiffrés au tarif > 650 km, distance « inconnue » dans la sortie 1")
def _():
    camions = P[P["supplier_id"].isin(couts.SANS_DISTANCE)]
    assert len(camions) > 0, "aucun fournisseur sans distance dans le planning"
    poids = camions["poids"].fillna(couts.POIDS_PAR_DEFAUT)
    assert np.allclose(camions["cout"], [couts.cout_transport(w, couts.DISTANCE_PAR_DEFAUT) for w in poids])
    assert r.sortie1.loc[r.sortie1["fournisseur"].isin(couts.SANS_DISTANCE), "distance_km"].isna().all()


@test("1.3", "Début du planning : tous les camions partent un jour fixe ; ceux d'avant le 09/11 partent dans la semaine ajoutée "
             "(à partir du lundi 02/11), au plus 7 jours avant le besoin, et ne livrent que des besoins de la première semaine")
def _():
    opti = P[~P["reserve"]]
    hors = [d.day_of_week not in prof for d, prof in zip(opti["date"], opti["profil"])]
    assert not any(hors), f"{sum(hors)} camion(s) hors de leurs jours fixes"
    avant = opti[opti["date"] < donnees.DEBUT]
    assert len(avant) > 0 and (avant["date"] >= donnees.DEBUT_LIVRAISONS).all()
    lignes_avant = L[L["camion_id"].isin(avant["camion_id"])]
    assert (lignes_avant["need_date"] < donnees.DEBUT + pd.Timedelta(days=7)).all(), "besoin d'après la première semaine"
    assert lignes_avant["avance_jours"].between(0, 7).all()


@test("1.3", "Cas impossibles notifiés : chaque jour où un couple reçoit 2 camions ou plus, et chaque camion réservé hors des "
             "jours fixes du couple, figure dans les alertes avec sa cause et l'action à mener")
def _():
    a = r.alertes
    c9 = contraintes.c9_un_camion_par_jour(P, L)
    assert c9["justifie"].all(), "2 camions un même jour sans cause"
    a9 = a[a["regle"].str.startswith("C9")]
    assert set(zip(c9["plant"], c9["supplier_id"], c9["date"])) == set(zip(a9["usine"], a9["fournisseur"], a9["date"]))
    profils = P[~P["reserve"]].groupby(["plant", "supplier_id"])["profil"].first()
    reserves = P[P["reserve"]]
    hors = sum(profils.get((p, s)) is None or d.day_of_week not in profils.get((p, s))
               for p, s, d in zip(reserves["plant"], reserves["supplier_id"], reserves["date"]))
    assert (a["regle"].str.startswith("C8")).sum() == hors, "camion réservé hors jours fixes non notifié"
    assert a["cause"].str.len().gt(0).all() and a["a_faire"].str.len().gt(0).all()


resultats("1.3")

### 1.4 Exhaustivité des matériaux livrés et anticipation des besoins

le moteur réorganise les livraisons (il regroupe des besoins de plusieurs jours dans un même camion), mais il ne doit rien perdre, rien inventer et ne jamais mettre une usine en rupture. Concrètement :
- exhaustivité : chaque ligne de besoin du planning d'entrée (fichiers silver et quarantaine) se retrouve dans un camion, une seule fois ; une ligne oubliée serait une pièce jamais livrée, une ligne en double un camion chargé pour rien 

- matériaux : pour chaque material, on compare la quantité demandée par l'usine (planning d'entrée) à la quantité chargée dans les camions (sortie 2). Le regroupement change les jours et les camions, mais pas les quantités : les deux doivent être égales 

- anticipation : chaque ligne arrive au plus tard le jour de son besoin (pas de rupture à l'usine), et au plus 7 jours avant (l'usine n'a pas à stocker plus d'une semaine).

Le graphique montre avec combien de jours d'avance chaque ligne est livrée : une barre par nombre de jours, de « le jour même » à « 7 jours avant », la limite autorisée. Un camion part un des jours fixes du couple et emporte les besoins des jours suivants : plus le besoin est loin du jour fixe, plus il est livré tôt.

In [ ]:
# ---------- Ce qui est comparé ----------
CLES = ["plant", "supplier_id", "material", "unloading_point"]
entree = pd.concat([donnees.besoins, donnees.lignes_reserves])                          # planning d'entrée : besoins à livrer
demande = entree.groupby(CLES)["schedule_quantity"].sum()                   # quantité demandée par material
livre = (r.sortie2.rename(columns={"usine": "plant", "fournisseur": "supplier_id", "quai": "unloading_point"})
         .groupby(CLES)["quantite"].sum())                                  # quantité chargée dans les camions (sortie 2)


@test("1.4", f"Aucune ligne de besoin oubliée ni livrée deux fois : les {len(donnees.IDS_ATTENDUS)} lignes du planning d'entrée "
             f"sont chacune dans un camion, une seule fois, et la sortie 2 les compte toutes")
def _():
    assert L["id_ligne"].is_unique and set(L["id_ligne"]) == donnees.IDS_ATTENDUS and L["camion_id"].notna().all()
    assert r.sortie2["nb_lignes"].sum() == len(donnees.IDS_ATTENDUS) and set(r.sortie2["camion_id"]) == set(P["camion_id"])


@test("1.4", f"Chaque material est livré en quantité exacte : pour les {len(demande)} combinaisons material × quai × couple, "
             f"quantité demandée (planning d'entrée) = quantité chargée dans les camions (sortie 2), au total "
             f"{formatage.nombre(demande.sum())} pièces des deux côtés")
def _():
    ecarts = demande.sub(livre, fill_value=0).abs()
    assert (ecarts < 1e-6).all(), f"{(ecarts >= 1e-6).sum()} materials avec un écart"


@test("1.4", f"Aucune usine en rupture, aucun stock de plus d'une semaine : pour les {len(L)} lignes, date de livraison "
             f"entre 7 jours avant et le jour du besoin (avance de {L['avance_jours'].min()} à {L['avance_jours'].max()} jours)")
def _():
    assert L["avance_jours"].between(0, 7).all(), L["avance_jours"].describe().to_string()


tests_14 = pd.DataFrame([r for r in RESULTATS.values() if r["Section"] == "1.4"])
tests_14[["Ce qu'on veut garantir", "Comment on le vérifie"]] = tests_14["Test"].str.split(" : ", n=1, expand=True)
display(tests_14[["Ce qu'on veut garantir", "Comment on le vérifie", "Résultat"]].style.hide(axis="index")
        .set_properties(**{"text-align": "left", "white-space": "normal"})
        .map(lambda v: "color: #b00020; font-weight: bold" if v in ("ÉCHEC", "ERREUR") else "", subset=["Résultat"]))

# ---------- Avec combien de jours d'avance les usines reçoivent-elles leurs besoins ? ----------
import matplotlib.pyplot as plt

avance = L["avance_jours"].value_counts().sort_index().reindex(range(8), fill_value=0)
part = avance / avance.sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
barres = ax.bar(avance.index, part * 100, color="royalblue", edgecolor="black")
ax.bar_label(barres, labels=[f"{formatage.pourcent(p)}\n({n} lignes)" for p, n in zip(part, avance)], fontsize=8, padding=2)
ax.set_xticks(range(8), ["le jour\nmême"] + [f"{j} j avant" for j in range(1, 8)])
ax.set_xlabel("Date de livraison par rapport à la date du besoin")
ax.set_ylabel("Part des lignes de besoin (%)")
ax.set_title(f"Avance des livraisons sur le besoin ({len(L)} lignes) : jamais de retard, jamais plus de 7 jours")
ax.margins(y=0.2)
ax.grid(axis="y", alpha=0.3)
plt.show()

### 1.5 Résultats des tests

In [ ]:
bilan_tests = pd.DataFrame(RESULTATS.values())
synthese = (bilan_tests.groupby("Section", sort=False)["Résultat"].value_counts().unstack(fill_value=0)
            .reindex(columns=["OK", "ÉCHEC", "ERREUR"], fill_value=0))
synthese.insert(0, "Tests", synthese.sum(axis=1))
synthese.index = synthese.index.map({"1.1": "1.1 Tests unitaires", "1.2": "1.2 Respect des contraintes",
                                     "1.3": "1.3 Cas limites", "1.4": "1.4 Exhaustivité et anticipation"})
synthese.loc["Total"] = synthese.sum()
display(synthese)

echecs = bilan_tests[bilan_tests["Résultat"] != "OK"]
print("Tous les tests passent." if echecs.empty else f"{len(echecs)} test(s) en échec :")
echecs if len(echecs) else None

Résultat : les 17 tests passent.
- Les fonctions clés du moteur donnent les résultats attendus sur des cas construits à la main, y compris aux bornes : tranche exacte, 7 jours d'avance acceptés et 8 refusés, capacité atteinte en poids ou en volume, profil à 2 jours utilisé ou non. Le prix calculé retrouve l'exemple de la rate sheet (335,34 €).

- Le planning final respecte toutes les contraintes dures définies au milestone 3, et le lissage reste à 2 semaines hors tolérance sur 20. Chaque fonction de vérification détecte l'erreur qu'on lui introduit : les contrôles du milestone 3 sont donc fiables.

- Les hypothèses sur les données manquantes sont appliquées comme prévu : la ligne de plus de 24,5 t est exclue, chaque ligne sans poids a son camion réservé, et les fournisseurs sans distance sont chiffrés au tarif le plus haut. Aucun camion ne part hors des jours fixes : les premiers besoins partent dans la semaine ajoutée avant le planning (à partir du 02/11). Les cas impossibles (lignes sans poids, besoin d'un jour > 24,5 t) sont tous notifiés dans les alertes.

- Rien n'est perdu ni créé : chaque ligne est livrée une fois, et la quantité livrée est égale à la quantité demandée pour chaque material, quai et couple. Aucune usine n'est livrée en retard. 13 % des lignes arrivent le jour de leur besoin et 16 % avec 7 jours d'avance, la limite autorisée : c'est le prix du regroupement sur un seul jour fixe par semaine pour presque tous les couples.

## 2. Analyse comparative (avant / après)

### Nouvelles distributions (taux de remplissage, coûts, ...)

### Étude des économies réalisées (réalisé vs potentiel)

## Conclusion